# Prompt 2 - Task-Splitting Prompting

This notebook contains the implementation of the second prompting strategy used in the homework. It evaluates Qwen on the reduced `dataset_30` benchmark using a task-splitting design, in which the model first identifies the relevant schema elements and then generates the final Cypher query.


In [ ]:
!pip -q install -U transformers accelerate sentencepiece gdown


In [ ]:
import os
import gdown

os.makedirs("/content/data", exist_ok=True)

LINK_DATASET_30 = "https://drive.google.com/uc?id=1IOvn9rx5-hFES6PU-3Ow8th5h-cl4z6v"
DATASET_PATH = "/content/data/dataset_30.csv"

gdown.download(LINK_DATASET_30, DATASET_PATH, quiet=False)
print("Dataset saved to:", DATASET_PATH)


In [ ]:
import json
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import torch
from transformers import pipeline

DATASET_PATH = "/content/data/dataset_30.csv"
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
SLEEP_SECONDS = 0.1

FULL_SCHEMA = 'Node properties:\nMovie {posterEmbedding: LIST, url: STRING, runtime: INTEGER, revenue: INTEGER, budget: INTEGER, plotEmbedding: LIST, imdbRating: FLOAT, released: STRING, countries: LIST, languages: LIST, plot: STRING, imdbVotes: INTEGER, imdbId: STRING, year: INTEGER, poster: STRING, movieId: STRING, tmdbId: STRING, title: STRING}\nGenre {name: STRING}\nUser {userId: STRING, name: STRING}\nActor {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nDirector {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nPerson {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nRelationship properties:\nRATED {rating: FLOAT, timestamp: INTEGER}\nACTED_IN {role: STRING}\nDIRECTED {role: STRING}\nThe relationships:\n(:Movie)-[:IN_GENRE]->(:Genre)\n(:User)-[:RATED]->(:Movie)\n(:Actor)-[:ACTED_IN]->(:Movie)\n(:Actor)-[:DIRECTED]->(:Movie)\n(:Director)-[:DIRECTED]->(:Movie)\n(:Director)-[:ACTED_IN]->(:Movie)\n(:Person)-[:ACTED_IN]->(:Movie)\n(:Person)-[:DIRECTED]->(:Movie)'

SYSTEM_PROMPT = (
    "You are an expert Neo4j and Cypher assistant. "
    "Always answer with a single valid JSON object and nothing else. "
    "Never use SQL syntax such as GROUP BY, HAVING, JOIN, or SELECT. "
    "Use Cypher WITH for aggregation steps. "
    "Use only schema-valid labels, relationship types, and properties."
)

PIPE = None

def load_generation_pipeline(model_name: str = MODEL_NAME):
    global PIPE
    if PIPE is not None:
        return PIPE

    torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    PIPE = pipeline(
        "text-generation",
        model=model_name,
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    if PIPE.tokenizer.pad_token_id is None:
        PIPE.tokenizer.pad_token_id = PIPE.tokenizer.eos_token_id
    return PIPE

def call_model(prompt: str, max_new_tokens: int) -> str:
    pipe = load_generation_pipeline()
    outputs = pipe(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
        pad_token_id=pipe.tokenizer.pad_token_id,
    )
    generated = outputs[0]["generated_text"]
    text = generated[-1]["content"].strip() if isinstance(generated, list) else str(generated).strip()
    time.sleep(SLEEP_SECONDS)
    return text

def extract_json_block(text: str) -> Optional[Dict[str, Any]]:
    if not isinstance(text, str):
        return None

    cleaned = text.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        parsed = json.loads(cleaned)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        pass

    start = cleaned.find("{")
    if start == -1:
        return None

    depth = 0
    for index in range(start, len(cleaned)):
        char = cleaned[index]
        if char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                candidate = cleaned[start : index + 1]
                try:
                    parsed = json.loads(candidate)
                    return parsed if isinstance(parsed, dict) else None
                except json.JSONDecodeError:
                    return None
    return None

def normalize_cypher(text: Any) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*,\s*", ", ", text)
    return text

def read_dataset(dataset_path: str = DATASET_PATH) -> pd.DataFrame:
    df = pd.read_csv(dataset_path)
    required = {"id", "question", "gold_cypher"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")
    return df

def get_completed_ids(output_path: str) -> set[str]:
    path = Path(output_path)
    if not path.exists():
        return set()
    df = pd.read_csv(path)
    if "id" not in df.columns:
        return set()
    return set(df["id"].astype(str))

def append_result(output_path: str, result: Dict[str, Any]) -> None:
    row_df = pd.DataFrame([result])
    header = not Path(output_path).exists()
    row_df.to_csv(output_path, mode="a", header=header, index=False)


In [ ]:
OUTPUT_PATH = "/content/results_prompt2_qwen_dataset30.csv"
STEP1_MAX_NEW_TOKENS = 120
STEP2_MAX_NEW_TOKENS = 220
SCHEMA_NODE_LABELS = ["Movie", "Genre", "User", "Actor", "Director", "Person"]
SCHEMA_REL_TYPES = ["IN_GENRE", "RATED", "ACTED_IN", "DIRECTED"]

def build_step1_prompt(question: str) -> str:
    return f"""
Task:
Identify the schema elements that are required to answer the question.

Instructions:
- Choose only from the provided schema inventory.
- Prefer the minimal sufficient set.
- Do not explain outside JSON.

Available node labels:
{", ".join(SCHEMA_NODE_LABELS)}

Available relationship types:
{", ".join(SCHEMA_REL_TYPES)}

Question:
{question}

Return JSON:
{{
  "relevant_nodes": ["Movie"],
  "relevant_relationships": ["RATED"],
  "relevant_properties": ["runtime", "rating"]
}}
""".strip()

def build_step2_prompt(question: str, step1_payload: Dict[str, Any]) -> str:
    hints = json.dumps(step1_payload, ensure_ascii=True)
    return f"""
Task:
Generate the final Cypher query using the schema and the selected hints.

Instructions:
- Use only labels, relationships, and properties present in the schema.
- Do not invent schema elements.
- Do not use SQL keywords such as GROUP BY, HAVING, JOIN, or SELECT.
- If aggregation is needed, use Cypher WITH.
- Do not explain outside JSON.

Schema:
{FULL_SCHEMA}

Selected schema hints:
{hints}

Question:
{question}

Return JSON:
{{
  "reasoning": "short explanation",
  "cypher": "final Cypher query"
}}
""".strip()

def run_prompt2(resume: bool = True) -> pd.DataFrame:
    df = read_dataset()
    completed_ids = get_completed_ids(OUTPUT_PATH) if resume else set()

    for _, row in df.iterrows():
        if str(row["id"]) in completed_ids:
            continue

        step1_raw = ""
        step1_parse_ok = False
        relevant_nodes = []
        relevant_relationships = []
        relevant_properties = []
        step1_error = ""

        try:
            step1_raw = call_model(build_step1_prompt(row["question"]), max_new_tokens=STEP1_MAX_NEW_TOKENS)
            parsed1 = extract_json_block(step1_raw)
            if parsed1 is None:
                step1_error = "Could not parse Step 1 output as JSON."
            else:
                relevant_nodes = parsed1.get("relevant_nodes", [])
                relevant_relationships = parsed1.get("relevant_relationships", [])
                relevant_properties = parsed1.get("relevant_properties", [])
                step1_parse_ok = isinstance(relevant_nodes, list)
                if not step1_parse_ok:
                    step1_error = "Step 1 JSON parsed but relevant_nodes is not a list."
        except Exception as exc:
            step1_error = str(exc)

        step2_raw = ""
        step2_parse_ok = False
        reasoning = ""
        predicted_cypher = ""
        step2_error = ""

        if step1_parse_ok:
            step1_payload = {
                "relevant_nodes": relevant_nodes,
                "relevant_relationships": relevant_relationships,
                "relevant_properties": relevant_properties,
            }
            try:
                step2_raw = call_model(
                    build_step2_prompt(row["question"], step1_payload),
                    max_new_tokens=STEP2_MAX_NEW_TOKENS,
                )
                parsed2 = extract_json_block(step2_raw)
                if parsed2 is None:
                    step2_error = "Could not parse Step 2 output as JSON."
                else:
                    reasoning = str(parsed2.get("reasoning", "")).strip()
                    predicted_cypher = str(parsed2.get("cypher", "")).strip()
                    step2_parse_ok = bool(predicted_cypher)
                    if not step2_parse_ok:
                        step2_error = "Step 2 JSON parsed but cypher field is empty."
            except Exception as exc:
                step2_error = str(exc)
        else:
            step2_error = "Skipped Step 2 because Step 1 failed."

        result = {
            "id": row["id"],
            "difficulty": row.get("difficulty", ""),
            "source_type": row.get("source_type", ""),
            "source_row": row.get("source_row", ""),
            "question": row["question"],
            "gold_cypher": row.get("gold_cypher", ""),
            "prompt_name": "prompt_2_task_splitting_qwen",
            "model_name": MODEL_NAME,
            "step1_parse_ok": step1_parse_ok,
            "relevant_nodes": json.dumps(relevant_nodes, ensure_ascii=True),
            "relevant_relationships": json.dumps(relevant_relationships, ensure_ascii=True),
            "relevant_properties": json.dumps(relevant_properties, ensure_ascii=True),
            "step1_raw": step1_raw,
            "step1_error": step1_error,
            "step2_parse_ok": step2_parse_ok,
            "reasoning": reasoning,
            "predicted_cypher": predicted_cypher,
            "exact_match": normalize_cypher(predicted_cypher) == normalize_cypher(row["gold_cypher"]),
            "step2_raw": step2_raw,
            "step2_error": step2_error,
        }
        append_result(OUTPUT_PATH, result)
        print(f"[{row['id']}] step1_ok={step1_parse_ok} step2_ok={step2_parse_ok} exact_match={result['exact_match']}")

    return pd.read_csv(OUTPUT_PATH)

df_results = run_prompt2(resume=True)
df_results.head()
